source ~/miniconda3/etc/profile.d/conda.sh && conda activate moai_project


nlp/sentiment.py - how it works

1. truncation=True — automatically cuts articles that are too long for BERT's 512-token limit, so it won't crash on long texts
2. self.pipe(text) i analyze the text of the article
3. analyze_batch(texts) - jumps over multiple texts and returns a table
4. aggregate_daily(df, date_col) - 
            avg_sentiment — mean of all article scores that day
            sentiment_std — how much articles disagreed that day (high std = mixed news)
            num_articles — article count per day (useful for the "volume spike" contrarian signal)
5. if __name__ == "__main__": - Runs when execute python sentiment.py directly. Tests on 5 sample headlines and prints results

Then you can test the sentiment pipeline:


cd src && python -m nlp.sentiment

---
# NewsAPI - collecting articles for sentiment analysis

**What is it?** NewsAPI (newsapi.org) is a REST API to search millions of news articles from 150,000+ sources.

**Why we use it:** We need real financial articles (titles + descriptions) to run FinBERT on. NewsAPI gives us structured data with dates, which we need for daily sentiment aggregation.

**Free tier limits:**
- 100 requests/day, 100 articles per request
- Articles go back 1 month
- Requires API key (get one at newsapi.org/register)

**What it returns per article:**
- `title` — headline (what we feed to FinBERT)
- `description` — short summary (can also be analyzed)
- `publishedAt` — date (needed for daily aggregation)
- `source` — which outlet published it
- `url` — link to full article

## Setup
Install: `pip install newsapi-python`
Get API key at newsapi.org/register

## How collect_news.py works
- Uses NewsApiClient to fetch articles by keyword (e.g. "SPY OR S&P 500")
- API key loaded from .env (gitignored)
- Saves title, description, date, source, url to data/raw/spy_newsapi_articles.csv
- Run: `python -m src.data.collect_news`

## Cleaning articles (clean_articles.py)
Filters out junk from raw NewsAPI data.
- Must mention SPY/S&P/Dow/Nasdaq directly, OR have 2+ finance keywords
- Blacklists obvious junk (casinos, sports, golf, etc.)
- Drops: empty titles, duplicates, non-English
- 100 raw → ~6 clean articles (strict filter)
- Run: `python -m src.data.clean_articles`
- Input: data/raw/spy_newsapi_articles.csv → Output: data/processed/spy_articles_clean.csv

## Context chunking (summarizer.py)
FinBERT has a 512 token limit. For long articles we:
1. Scrape full article text from URL using newspaper3k
2. Use TextRank (sumy) to extract the most important sentences that fit in ~480 tokens
3. If scraping fails (paywall etc.) → falls back to title + description
- Install: `pip install newspaper3k sumy langdetect`
- Also needed: `python -c "import nltk; nltk.download('punkt_tab')"`

## Running sentiment on articles (run_sentiment.py)
Full pipeline: loads clean articles → fetches & summarizes each → runs FinBERT → saves scores.
- Run: `python -m src.nlp.run_sentiment`
- Output: data/processed/sentiment_scores.csv

## Current status
Pipeline works end-to-end: collect → clean → summarize → score → save.
Problem: only 6 articles survived strict cleaning from 1 month of NewsAPI free tier. Need more data for Granger causality.

## Daily collection workflow
NewsAPI free tier = 100 requests/day, so we accumulate articles over multiple days.
- `collect_news.py` now **appends** to existing CSV and dedupes by URL.
- Each day: run `collect_news.py` → `clean_articles.py` → `run_sentiment.py`.
- Day 2 (2026-04-09): 96 new articles → 195 total raw → 18 clean → sentiment scored across 2 dates.

## Added metadata columns (for Kacper's analysis)
`sentiment_scores.csv` now includes:
- `datetime` — full ISO timestamp (date + hour) from NewsAPI's publishedAt
- `tickers` — comma-separated tickers/companies detected in the article (e.g. "SPY,DJIA,TSLA")
- `title` — original article title for context

**Ticker detection:** regex with word boundaries on title + description. Currently recognizes SPY, DJIA, QQQ, IXIC, TSLA, AAPL, MSFT, GOOGL, AMZN, NVDA, META. Extend the `TICKER_ALIASES` dict in clean_articles.py to add more.

**Note:** articles collected before this change don't have full timestamps (only date). New collections from now on will include the hour.